In [ ]:
[
    {
      "id": "wavetable001",
      "name": "Dreamy Vaporwave Pad",
      "instrument": "wavetable",
      "keywords": {
        "soundType": ["pad"],
        "character": ["ethereal", "airy", "breathy"],
        "genre": ["vaporwave", "ambient"],
        "mood": ["dreamy", "atmospheric"],
        "technique": ["wavetable"],
        "modulation": ["evolving"],
        "inspiration": ["2000s video games"]
      },
      "fullDescription": ""
    },
    ...
  ]




In [26]:
import pandas as pd

wt_data = pd.read_csv('/Users/jake/Source/muse-art/polyhymnia/Data Labeling - AB WT Defaults.csv')

In [29]:
df = wt_data.drop(columns=['Completion'])
df.rename(columns={'Title': 'name', 'Prompt': 'fullDescription', 'Unnamed: 0': 'Type'}, inplace=True)


In [34]:
df = df[df['Type'].isin(['Pads', 'Malles', 'Mallets', 'Ambient & Evolving'])]
df['Type'] = df['Type'].replace('Malles', 'Mallets')


In [36]:
df.to_csv('chord-preset-descriptions.csv', index=False)

In [37]:
import json

# Assuming df has been created in the previous cell and contains the necessary columns
result = []

for index, row in df.iterrows():





    entry = {
        "id": f"wavetable{index + 1:03d}",  # Generating a unique ID for each entry
        "name": row['name'],
        "instrument": "wavetable",  # Assuming all entries are of the same instrument type
        "keywords": {
            "soundType": [],  # This should be filled based on the description
            "character": [],  # This should be filled based on the description
            "genre": [],  # This should be filled based on the description
            "mood": [],  # This should be filled based on the description
            "technique": ["wavetable"],  # Assuming this is constant
            "modulation": [],  # This should be filled based on the description
            "inspiration": []  # This should be filled based on the description
        },
        "fullDescription": row['fullDescription']
    }

    
    
    # Here you would implement logic to fill the keywords based on the fullDescription
    # For example, you could use some NLP techniques or keyword extraction methods

    result.append(entry)

# Print the result as a JSON list
print(json.dumps(result, indent=2))


[
  {
    "id": "wavetable001",
    "name": "Analog Soft Pad",
    "instrument": "wavetable",
    "keywords": {
      "soundType": [],
      "character": [],
      "genre": [],
      "mood": [],
      "technique": [
        "wavetable"
      ],
      "modulation": [],
      "inspiration": []
    },
    "fullDescription": "I want a soft, analog pad with a lush and atmospheric sound, reminiscent of 80's dark wave or post-punk synths, or the soundtrack to an 80's movie. The pad should have gentle attack, slightly extended decay, high sustain, and moderate release to create a sense of depth and warmth, and add a cinematic feel to the track. There should be a small amount of modulation to the wavetable and filter to create a subtle sense of movement, but not too much to distract the listener."
  },
  {
    "id": "wavetable002",
    "name": "Airlite",
    "instrument": "wavetable",
    "keywords": {
      "soundType": [],
      "character": [],
      "genre": [],
      "mood": [],
      "tec

In [38]:
len(result)

21

In [39]:
sys_prompt = """
Use the full description and title to label the sound with keywords. Here is an example of the expected JSON format:


    {
      "id": "wavetable001",
      "name": "Dreamy Vaporwave Pad",
      "instrument": "wavetable",
      "keywords": {
        "soundType": ["pad"],
        "character": ["ethereal", "airy", "breathy"],
        "genre": ["vaporwave", "ambient"],
        "mood": ["dreamy", "atmospheric"],
        "technique": ["wavetable"],
        "modulation": ["evolving"],
        "inspiration": ["2000s video games"]
      },
      "fullDescription": ""
    }

    The user will now provide you with a template of the JSON file for each preset, with some fields already filled in. Please complete the keywords based on the full description and title provided.

"""


In [40]:
import os
from openai import OpenAI

def create_openai_client():
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable is not set")
    return OpenAI(api_key=api_key)


In [41]:
from pydantic import BaseModel

class Keywords(BaseModel):
    soundType: list[str]
    character: list[str]
    genre: list[str]
    mood: list[str]
    technique: list[str]
    modulation: list[str]
    inspiration: list[str]
    




In [43]:
import json 
from tqdm import tqdm

completed_entries = []

for entry in tqdm(result):

    prompt = json.dumps(entry)


    client = create_openai_client()
    chat = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user', 'content': prompt}
        ],
        temperature=1.0,
        response_format=Keywords
    )

    content = json.loads(chat.choices[0].message.content)
    entry['keywords'] = content

    completed_entries.append(entry)


100%|██████████| 21/21 [00:46<00:00,  2.20s/it]


In [45]:
json.dump(completed_entries, open('wavetable-v01.json', 'w'), indent=2)